<a href="https://colab.research.google.com/github/minidiablo05/-/blob/main/cleaning_dividing_clean_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 1 (загрузка)

In [ ]:
import numpy as np
import pandas as pd
import re

df = pd.read_excel('датасетA_check.xlsx', sheet_name = 'Лист2')

In [ ]:
# 2. Переименовывание столбцов в латиницу
df = df.rename(columns={
    'возраст': 'age',
    'отсутствие беременности (год)': 'years_without_pregnancy',
    'метолы ВРТ в анамнезе (кол-во попыток)': 'art_attempts',
    'аллергия (0-нет,1-есть)': 'allergy',
    'ИМТ': 'bmi',
    'беременности,было': 'pregnancies_count',
    'выкидыши': 'miscarriages',
    'роды, было': 'deliveries',
    'менструации с': 'menarche_age',
    'Установились (сразу-1, нет-0)': 'menstruation_established',
    'Регулярные': 'cycle_regular',
    'по кол-во дней': 'cycle_days',
    'через кол-во дней': 'cycle_interval',
    'Объем 1-умеренные 0-скудные 2-обильные': 'bleeding_volume',
    'Болезненные 1-да 0-нет': 'painful_periods',
    'Нарушение менструального цикла': 'cycle_disorder',
    'Половая жизнь': 'sexual_life_start',
    'Супруг, возраст': 'partner_age',
    'Наследственность': 'heredity',
    'заболевания (по серьезности болезни)': 'disease_severity',
    'СПКЯ': 'pcos',
    'наличие миомы': 'myoma',
    'мужской фактор бесплодия (1-да, 0-женское)': 'male_factor',
    'АМГ': 'amh',
    'ФСГ': 'fsh',
    'Получено': 'oocytes_retrieved',
    'перенос эмбриона': 'embryo_transfer',
    'ЭКО(1) / ИКСИ(2)': 'ivf_icsi',
    'Криоконсервация (1-да, 0-нет)': 'cryopreservation',
    'беременность (1-наступила, 0-нет)': 'pregnancy',
    'роды': 'delivery'
})
df.columns

In [ ]:
df.info()

In [ ]:
# подсчет общей доли пропусков
total_cells = df.size
missing_cells = df.isnull().sum().sum()
missing_pct = (missing_cells / total_cells) * 100

print(f"Общая доля пропусков: {missing_pct:.2f}%")

In [ ]:
df.head(20)

In [ ]:
# 3. Функция обработки диапазонов
def parse_range(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().replace(' ', '')
    # Ищем диапазон "число-число"
    match = re.search(r'(\d+)-(\d+)', s)
    if match:
        low = float(match.group(1))
        high = float(match.group(2))
        return (low + high) / 2.0
    try:
        return float(s)
    except ValueError:
        return np.nan

In [ ]:
df = df.replace(['-'], np.nan)

In [ ]:
df.loc[45: 65, 'cycle_interval']

In [ ]:
# Дополнительно: создаём признак научной новизны
df['cycle_range_reported'] = df['cycle_interval'].apply(lambda x: 1 if pd.notna(x) and '-' in str(x) else 0).astype('Int64')

In [ ]:
# 4. Применяем parse_range **ДО** замены на NaN (самое важное изменение!)
range_cols = ['cycle_days', 'cycle_interval', 'sexual_life_start', 'cycle_disorder', 'heredity', 'menstruation_established']

for col in range_cols:
    if col in df.columns:
        df[col] = df[col].apply(parse_range)

In [ ]:
# 5. Теперь безопасно заменяем оставшиеся заглушки
df = df.replace(['', ' ', 'NaN', 'nan', None], np.nan)

# 6. Удаляем полностью пустые строки
df = df.dropna(subset=df.columns.difference(['cycle_range_reported']), thresh=2)
df = df.reset_index(drop=True)

# 7. Приводим все числовые столбцы к float
numeric_cols = ['age', 'years_without_pregnancy', 'art_attempts', 'allergy', 'bmi',
                'pregnancies_count', 'miscarriages', 'deliveries', 'menarche_age',
                'cycle_days', 'cycle_interval', 'bleeding_volume', 'painful_periods',
                'sexual_life_start', 'partner_age', 'disease_severity', 'amh', 'fsh',
                'oocytes_retrieved', 'embryo_transfer', 'ivf_icsi', 'cryopreservation',
                'pregnancy', 'delivery']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
df.loc[df['pregnancy'].isna() & (df['delivery'] == 1), 'pregnancy'] = 1

In [ ]:
# 9. Финальная проверка и сохранение
print("=== ФИНАЛЬНЫЙ РЕЗУЛЬТАТ ПОСЛЕ ИСПРАВЛЕНИЯ ===")
print(df.shape)
print(df.info())

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df[['cycle_days', 'cycle_interval', 'cycle_range_reported']].head(20)

In [ ]:
df.loc[45: 65, 'cycle_days']

In [ ]:
df.loc[45: 65, 'cycle_interval']

In [ ]:
df.loc[45: 65, 'cycle_range_reported']

In [ ]:
df.to_excel('dataset_cleaned_step1_fixed.xlsx', index=False)

In [ ]:
df.shape

In [ ]:
total_cells = df.size
missing_cells = df.isnull().sum().sum()
missing_pct = (missing_cells / total_cells) * 100

print(f"Общая доля пропусков: {missing_pct:.2f}%")

# Этап 2 (Разделение на delivery и pregnancy)

In [ ]:
# 1. Создаём копии
df_main = df.copy()   # для основной модели (delivery)
df_preg = df.copy()   # для дополнительного эксперимента (pregnancy)

In [ ]:
# 3. Удаляем строки, где нет целевой переменной
df_main = df_main.dropna(subset=['delivery']).reset_index(drop=True)
df_preg = df_preg.dropna(subset=['pregnancy']).reset_index(drop=True)

In [ ]:
# 5. Проверяем результат
print("=== df_main (delivery) ===")
print("Shape:", df_main.shape)
print("delivery non-null:", df_main['delivery'].notna().sum())
print("pregnancy non-null:", df_main['pregnancy'].notna().sum())

print("\n=== df_preg (pregnancy) ===")
print("Shape:", df_preg.shape)
print("pregnancy non-null:", df_preg['pregnancy'].notna().sum())

In [ ]:
# 6. Сохраняем
df_main.to_excel('dataset_main_delivery_v2.xlsx', index=False)
df_preg.to_excel('dataset_pregnancy_v2.xlsx', index=False)

In [ ]:
df_preg.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
df_main.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# Делаем подсчет пар
df_main[['pregnancy', 'delivery']].value_counts(dropna=False)

In [ ]:
df_main.info()

# Этап 3 (обработка DELIVERY)

In [ ]:
df_main.info()

In [ ]:
df_main.head()

In [ ]:
df_main[['allergy', 'pcos', 'myoma', 'male_factor']].isnull().sum().sort_values(ascending=False)

In [ ]:
# 1. Простые заполнения (мода / 0)
df_main['male_factor'] = df_main['male_factor'].fillna(0)

In [ ]:
from sklearn.impute import KNNImputer

# 2. KNNImputer для клинически связанных признаков (рекомендуется для медицинских данных)
impute_cols = ['bmi', 'amh', 'fsh', 'cycle_days', 'cycle_interval',
               'sexual_life_start', 'partner_age', 'disease_severity']

imputer = KNNImputer(n_neighbors=5)
df_main[impute_cols] = imputer.fit_transform(df_main[impute_cols])

In [ ]:
# 3. Feature Engineering (научная новизна)
df_main['age_group'] = pd.cut(df_main['age'], bins=[0, 30, 35, 40, 50],
                              labels=[0, 1, 2, 3])
df_main['bmi_category'] = pd.cut(df_main['bmi'], bins=[0, 18.5, 25, 30, 100],
                                 labels=[0, 1, 2, 3])
df_main['amh_fsh_ratio'] = df_main['amh'] / (df_main['fsh'] + 0.001)   # защита от деления на 0
df_main['total_art_attempts_group'] = pd.cut(df_main['art_attempts'],
                                             bins=[-1, 0, 2, 5, 10],
                                             labels=[0, 1, 2, 3])
df_main['poor_responder'] = (df_main['oocytes_retrieved'] < 5).astype(int)  # клинический маркер


In [ ]:
# 4. Финальная проверка
print("=== df_main после Этапа 3 ===")
print(df_main.shape)
print(df_main.isnull().sum().sort_values(ascending=False).head(25))

In [ ]:
# 5. Сохраняем
df.to_excel('dataset_main_delivery_final.xlsx', index=False)

### Проверка

In [ ]:
df_main.shape

In [ ]:
# Таблица Feature Engineering (научная новизна)
df_main[['age_group', 'bmi_category', 'amh_fsh_ratio', 'total_art_attempts_group', 'poor_responder']].head()

In [ ]:
df_main.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
df_main.head()

In [ ]:
df_main[['age', 'years_without_pregnancy', 'art_attempts', 'bmi',
                    'pregnancies_count', 'miscarriages', 'deliveries', 'menarche_age',
                    'cycle_days', 'cycle_interval', 'sexual_life_start', 'partner_age',
                    'disease_severity', 'amh', 'fsh', 'embryo_transfer', 'ivf_icsi',
                    'cryopreservation', 'amh_fsh_ratio']].isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
df_main[['age_group', 'bmi_category', 'total_art_attempts_group',
                        'allergy', 'pcos', 'myoma', 'male_factor', 'heredity',
                        'cycle_disorder', 'cycle_range_reported', 'poor_responder']].isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# Быстрое исправление оставшихся NaN
df_main['art_attempts'] = df_main['art_attempts'].fillna(0)                    # клинически оправдано
df_main['miscarriages'] = df_main['miscarriages'].fillna(0)
df_main['deliveries'] = df_main['deliveries'].fillna(0)
df_main['menarche_age'] = df_main['menarche_age'].fillna(df_main['menarche_age'].median())
df_main['cryopreservation'] = df_main['cryopreservation'].fillna(0)                    # клинически оправдано
df_main['ivf_icsi'] = df_main['ivf_icsi'].fillna(0)
df_main['cycle_disorder'] = df_main['cycle_disorder'].fillna(0)               # 0 = нет нарушения

In [ ]:
df_main[['age', 'years_without_pregnancy', 'art_attempts', 'bmi',
                    'pregnancies_count', 'miscarriages', 'deliveries', 'menarche_age',
                    'cycle_days', 'cycle_interval', 'sexual_life_start', 'partner_age',
                    'disease_severity', 'amh', 'fsh', 'embryo_transfer', 'ivf_icsi',
                    'cryopreservation', 'amh_fsh_ratio']].isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
#  Пересоздаём группировку после заполнения
df_main['total_art_attempts_group'] = pd.cut(df_main['art_attempts'],
                                        bins=[-1, 0, 2, 5, 10],
                                        labels=[0, 1, 2, 3])

In [ ]:
df_main[['age_group', 'bmi_category', 'total_art_attempts_group',
                        'allergy', 'pcos', 'myoma', 'male_factor', 'heredity',
                        'cycle_disorder', 'cycle_range_reported', 'poor_responder']].isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
df_main[['age_group', 'bmi_category', 'amh_fsh_ratio', 'total_art_attempts_group', 'poor_responder']].head()

In [ ]:
df_main.info()

In [ ]:
df_main.shape

In [ ]:
df_main.to_excel('FINAL_delivery.xlsx', index=False)

# Этап 4 (обрабодка PREGNANCY)

In [ ]:
df_preg.head()

In [ ]:
df_preg.info()

In [ ]:
df_preg.isnull().sum().sort_values(ascending=False).head(25)

In [ ]:
df_preg[['allergy', 'pcos', 'myoma', 'male_factor']].isnull().sum().sort_values(ascending=False)

In [ ]:
df_preg['male_factor'] = df['male_factor'].fillna(0)

In [ ]:
from sklearn.impute import KNNImputer

# 2. KNNImputer для клинически связанных признаков (рекомендуется для медицинских данных)
impute_cols = ['bmi', 'amh', 'fsh', 'cycle_days', 'cycle_interval',
               'sexual_life_start', 'partner_age', 'disease_severity']

imputer = KNNImputer(n_neighbors=5)
df_preg[impute_cols] = imputer.fit_transform(df_preg[impute_cols])

In [ ]:
# 3. Feature Engineering (научная новизна)
df_preg['age_group'] = pd.cut(df_preg['age'], bins=[0, 30, 35, 40, 50],
                              labels=[0, 1, 2, 3])
df_preg['bmi_category'] = pd.cut(df_preg['bmi'], bins=[0, 18.5, 25, 30, 100],
                                 labels=[0, 1, 2, 3])
df_preg['amh_fsh_ratio'] = df_preg['amh'] / (df_preg['fsh'] + 0.001)   # защита от деления на 0
df_preg['total_art_attempts_group'] = pd.cut(df_preg['art_attempts'],
                                             bins=[-1, 0, 2, 5, 10],
                                             labels=[0, 1, 2, 3])
df_preg['poor_responder'] = (df_preg['oocytes_retrieved'] < 5).astype(int)  # клинический маркер


In [ ]:
# 4. Финальная проверка
print("=== df_preg после Этапа 1 ===")
print(df_preg.shape)
print(df_preg.isnull().sum().sort_values(ascending=False).head(25))

In [ ]:
df_preg.shape

In [ ]:
df_preg[['age_group', 'bmi_category', 'amh_fsh_ratio', 'total_art_attempts_group', 'poor_responder']].head()

In [ ]:
df_preg.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
df_preg.head()

In [ ]:
df_preg[['age', 'years_without_pregnancy', 'art_attempts', 'bmi',
                    'pregnancies_count', 'miscarriages', 'deliveries', 'menarche_age',
                    'cycle_days', 'cycle_interval', 'sexual_life_start', 'partner_age',
                    'disease_severity', 'amh', 'fsh', 'embryo_transfer', 'ivf_icsi',
                    'cryopreservation', 'amh_fsh_ratio']].isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
df_preg[['age_group', 'bmi_category', 'total_art_attempts_group',
                        'allergy', 'pcos', 'myoma', 'male_factor', 'heredity',
                        'cycle_disorder', 'cycle_range_reported', 'poor_responder']].isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# Быстрое исправление оставшихся NaN (до препроцессинга!)
df_preg['art_attempts'] = df_preg['art_attempts'].fillna(0)                    # клинически оправдано
df_preg['miscarriages'] = df_preg['miscarriages'].fillna(0)
df_preg['deliveries'] = df_preg['deliveries'].fillna(0)
df_preg['menarche_age'] = df_preg['menarche_age'].fillna(df_preg['menarche_age'].median())
df_preg['cryopreservation'] = df_preg['cryopreservation'].fillna(0)                    # клинически оправдано
df_preg['ivf_icsi'] = df_preg['ivf_icsi'].fillna(0)
df_preg['cycle_disorder'] = df_preg['cycle_disorder'].fillna(0)               # 0 = нет нарушения

In [ ]:
df_preg[['age', 'years_without_pregnancy', 'art_attempts', 'bmi',
                    'pregnancies_count', 'miscarriages', 'deliveries', 'menarche_age',
                    'cycle_days', 'cycle_interval', 'sexual_life_start', 'partner_age',
                    'disease_severity', 'amh', 'fsh', 'embryo_transfer', 'ivf_icsi',
                    'cryopreservation', 'amh_fsh_ratio']].isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# 3. Пересоздаём группировку после заполнения
df_preg['total_art_attempts_group'] = pd.cut(df_preg['art_attempts'],
                                        bins=[-1, 0, 2, 5, 10],
                                        labels=[0, 1, 2, 3])

In [ ]:
df_preg[['age_group', 'bmi_category', 'total_art_attempts_group',
                        'allergy', 'pcos', 'myoma', 'male_factor', 'heredity',
                        'cycle_disorder', 'cycle_range_reported', 'poor_responder']].isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
df_preg[['age_group', 'bmi_category', 'amh_fsh_ratio', 'total_art_attempts_group', 'poor_responder']].head()

In [ ]:
df_preg.info()

In [ ]:
df_preg.to_excel('test_pregnancy.xlsx', index=False)

### Добработка

In [ ]:
df_preg.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# 1. Удаляем столбцы с очень большим количеством пропусков (>50%)
cols_to_drop = ['oocytes_retrieved', 'menstruation_established',
                'bleeding_volume', 'painful_periods', 'cycle_regular']
df_preg = df_preg.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
df_preg.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# 3. delivery — заполняем 0 (нет информации о родах)
df_preg['delivery'] = df_preg['delivery'].fillna(0)

In [ ]:
df_preg.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
df_preg.info()

In [ ]:
df_preg.to_excel('FINAL_pregnancy.xlsx', index=False)

In [ ]:
df_preg.shape

In [ ]:
print("=" * 60)
print("БАЛАНС КЛАССОВ В ДАТАСЕТАХ")
print("=" * 60)

# Pregnancy
print("\n[ PREGNANCY ]")
print(df_preg['pregnancy'].value_counts())
print("Доля классов:")
print(df_preg['pregnancy'].value_counts(normalize=True).round(4))

# Delivery
print("\n[ DELIVERY ]")
print(df_main['delivery'].value_counts())
print("Доля классов:")
print(df_main['delivery'].value_counts(normalize=True).round(4))

# Соотношение положительного класса
print("\n[ СООТНОШЕНИЕ positive/negative ]")
preg_pos = df_preg['pregnancy'].mean()
del_pos = df_main['delivery'].mean()
print(f"Pregnancy positive class: {preg_pos:.3f} (соотношение 1:{(1-preg_pos)/preg_pos:.1f})")
print(f"Delivery positive class:  {del_pos:.3f} (соотношение 1:{(1-del_pos)/del_pos:.1f})")